# Colab Base para el Trabajo Práctico (versión 4)
Dada la diferencia que existe entre los dataset en la **geolocalización, operación, tipo de propiedad, moneda**, este filtro se basara en tales campos.
Imputación de lat y lon

In [1]:
import pandas as pd
import sqlite3

import sklearn as sk
from sklearn import model_selection
from sklearn import ensemble
from sklearn import metrics

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 0. Lectura de datos

In [3]:
# TODO: Cambiar para que apunte al directorio correcto
DIR = "/content/drive/MyDrive/UBA/Especializacion/DM/Data"

In [63]:
engine = sqlite3.connect(f"{DIR}/entrenamiento.db")

df_ent = pd.read_sql("SELECT * FROM entrenamiento", engine, index_col="id")
df_ent.head(3)

,description,address,lat,lon,publication_date,publisher_id,features,location_0,location_1,location_2,location_3,location_4,operation_type,property_type,source,price,currency_type
id,,,,,,,,,,,,,,,,,
0,galpón de 275 metros cuadrados cubiertos. ...,"Soldini, Rosario, Santa Fe, ARG",-33.008797,-60.743675,"Hace 1 semana, 17 horas",None,electricidad;cerca de:,Argentina,Santa Fe,Soldini,Rosario,None,venta,galpón,properati,150000.0,dolares
1,"venta de galpón en barrio urquiza, rosario sob...","Boulevard 27 de Febrero, Rosario, S2009, Santa...",-32.959099,-60.710079,1 ene 2026,None,1 baño;700 m²;electricidad;cerca de:,Argentina,Santa Fe,Rosario,Rosario,None,venta,galpón,properati,250000.0,dolares
2,mira esta y otras propiedades en nuestro sitio...,"Calle Virasoro 3450, Rosario, S2003, Santa Fe,...",-32.963959,-60.675930,30 nov 2025,None,2 dormitorios;1 baño;250 m²;internet;electrici...,Argentina,Santa Fe,Rosario,Rosario,None,venta,galpón,properati,130000.0,dolares


In [ ]:
# cantidad de filas y columnas
df_ent.shape

(1292674, 17)

In [ ]:
# lista de columnas del dataframe
df_ent.columns

Index(['description', 'address', 'lat', 'lon', 'publication_date',
       'publisher_id', 'features', 'location_0', 'location_1', 'location_2',
       'location_3', 'location_4', 'operation_type', 'property_type', 'source',
       'price', 'currency_type'],
      dtype='object')

In [5]:
# Dataset a predecir:
df_ap = pd.read_csv(f"{DIR}/a_predecir.csv", index_col="id")

## 1. Entender los datos (AID)

In [ ]:


print(f"Dataset a predecir - Operaciones:\n-{df_ap.operation_type.unique()}\n")
print(f"Dataset a entrenar - Operaciones:\n-{df_ent.operation_type.unique()}")
print("-"*90)

print(f"Dataset a predecir - Propiedad:\n-{df_ap.property_type.unique()}\n")
print(f"Dataset a entrenar - Propiedad:\n-{df_ent.property_type.unique()}")
print("-"*90)

print(f"Dataset a predecir - País:\n-{df_ap.location_0.unique()}\n")
print(f"Dataset a entrenar - País:\n-{df_ent.location_0.unique()}\n")
print("-"*90)
print(f"Dataset a predecir - Provincia:\n-{df_ap.location_1.unique()}\n")
print(f"Dataset a entrenar - Provincia:\n-{df_ent.location_1.unique()}\n")
print("-"*90)
print(f"Dataset a predecir - Ciudad:\n-{df_ap.location_2.unique()}\n")
print(f"Dataset a entrenar - Ciudad:\n-{df_ent.location_2.unique()}")
print("-"*90)

print(f"Dataset a predecir - Moneda:\n-{df_ap.currency_type.unique()}\n")
print(f"Dataset a entrenar - Moneda:\n-{df_ent.currency_type.unique()}\n")
print("-"*90)
print(f"Dataset a predecir - Fuente de datos:\n-{df_ap.source.unique()}\n")
print(f"Dataset a entrenar - Fuente de datos:\n-{df_ent.source.unique()}\n")
print("-"*90)

Dataset a predecir - Operaciones:
-['venta']

Dataset a entrenar - Operaciones:
-['venta' None 'alquiler' 'temporal' 'alquiler / temporal'
 'venta / temporal' 'venta / alquiler' 'venta / alquiler / temporal'
 'traspaso' 'renta' 'alquiler temporal' 'sin operacion']
------------------------------------------------------------------------------------------
Dataset a predecir - Propiedad:
-['casa' 'departamento' 'cochera']

Dataset a entrenar - Propiedad:
-['galpón' None 'casa' 'terreno' 'departamento' 'local comercial' 'oficina'
 'cochera' 'villa' 'terrenos' 'garage' 'oficina comercial' 'ph'
 'consultorio' 'bodega-galpón' 'hotel' 'edificio' 'depósito' 'campo'
 'quinta vacacional' 'fondo de comercio' 'desarrollo vertical'
 'desarrollo horizontal' 'bóveda, nicho o parcela' 'cama náutica'
 'casa-duplex' 'galpon' 'quinta' 'cabana' 'penthouse' 'loft'
 'fondo-de-comercio' 'inmueble-productivo' 'local' 'fabrica' 'nave'
 'deposito' 'oficinas' 'departamentos' 'casas' 'comercios' 'depósitos'
 'empr

## 2. Limpiar y transformar los datos (DM)

In [6]:
# Entrenamiento se basa en los parámentreos lon y lat, este filtro lo haremos por geolocalización,
#  operación, tipo de propiedad, moneda, fuente de datos
filtro = ((df_ent["location_0"] == 'Argentina') &
 (df_ent["location_1"].isin(['Ciudad Autónoma de Buenos Aires','Capital Federal'])) &
          (df_ent["operation_type"] == "venta") &
          (df_ent["property_type"].isin(['casa','departamento','cochera'])) &
          (df_ent["currency_type"].isin(['dolares','pesos'])))
df_ent = df_ent.loc[filtro]
df_ent.shape
# test = df_ent.loc[filtro]
# test.shape

(112831, 17)

In [7]:
# Guardo el dataframe en un csv, para luego acceder más facil.
df_ent.to_csv("/content/drive/MyDrive/UBA/Especializacion/DM/Data/raw/entrenamiento.csv")

In [51]:
# La creación de modelos requiere que no haya valores perdidos
# por eso llenamos todo con 0 a lo bestia
# TODO: mejorar la imputación de valores perdidos
df_ent = df_ent.fillna(0)

In [52]:
df_ent.info()

<class 'pandas.core.frame.DataFrame'>
Index: 112831 entries, 3659 to 1270469
Data columns (total 3 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   lat     112831 non-null  float64
 1   lon     112831 non-null  float64
 2   price   112831 non-null  float64
dtypes: float64(3)
memory usage: 3.4 MB


In [ ]:
df_ent["location_4"] =df_ent["location_4"].astype("object")
df_ent.info()

<class 'pandas.core.frame.DataFrame'>
Index: 45189 entries, 3659 to 696885
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   lat         45189 non-null  float64
 1   lon         45189 non-null  float64
 2   location_4  45189 non-null  object 
 3   price       45189 non-null  float64
dtypes: float64(3), object(1)
memory usage: 2.7+ MB


In [ ]:
df_ent["source"].value_counts()

,count
source,
argenprop,67642
properati,29692
zonaprop,14066


In [9]:
df_ent.loc[df_ent["lat"].isna(), "source"].value_counts()

,count
source,
argenprop,67631
zonaprop,13


In [13]:
df_ent.columns

Index(['description', 'address', 'lat', 'lon', 'publication_date',
       'publisher_id', 'features', 'location_0', 'location_1', 'location_2',
       'location_3', 'location_4', 'operation_type', 'property_type', 'source',
       'price', 'currency_type'],
      dtype='object')

In [26]:
# Guardo long y lat de aquellos no null.
ciudades_lat_lon = df_ent.loc[~df_ent["lat"].isna(), ["lat","lon","location_2"]].drop_duplicates("location_2")

In [29]:
df_ent2 = df_ent.copy()

In [39]:
for i in ciudades_lat_lon.index:
  df_ent2.loc[df_ent2["location_2"] == ciudades_lat_lon.loc[i,"location_2"], "lat" ]= ciudades_lat_lon.loc[i,"lat"]
  df_ent2.loc[df_ent2["location_2"] == ciudades_lat_lon.loc[i,"location_2"], "lon" ]= ciudades_lat_lon.loc[i,"lon"]

In [40]:
df_ent2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 112831 entries, 3659 to 1270469
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   description       112831 non-null  object 
 1   address           112536 non-null  object 
 2   lat               112831 non-null  float64
 3   lon               112831 non-null  float64
 4   publication_date  45189 non-null   object 
 5   publisher_id      83139 non-null   object 
 6   features          112716 non-null  object 
 7   location_0        112831 non-null  object 
 8   location_1        112831 non-null  object 
 9   location_2        112827 non-null  object 
 10  location_3        99351 non-null   object 
 11  location_4        7849 non-null    object 
 12  operation_type    112831 non-null  object 
 13  property_type     112831 non-null  object 
 14  source            111400 non-null  object 
 15  price             112771 non-null  float64
 16  currency_type     112

In [42]:
df_ent = df_ent2.copy()

In [47]:
X.info()

<class 'pandas.core.frame.DataFrame'>
Index: 112831 entries, 3659 to 1270469
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   lat     112831 non-null  float64
 1   lon     112831 non-null  float64
dtypes: float64(2)
memory usage: 2.6 MB


In [50]:
# X_train.info()
y_train.info()

<class 'pandas.core.series.Series'>
Index: 90264 entries, 718795 to 114226
Series name: price
Non-Null Count  Dtype  
--------------  -----  
90216 non-null  float64
dtypes: float64(1)
memory usage: 1.4 MB


## 3. Entrenamiento del modelos (AA) - ⛔⛔⛔ NO TOCAR ⛔⛔⛔

In [53]:
# La creación de modelos requiere que todo el dataframe sea numérico
# Me quedo con las columnas numéricas solamente
# TODO: traducir las columnas con datos no numéricos a numéricos para que mejoren los modelos
df_ent = df_ent.select_dtypes('number')

X = df_ent[df_ent.columns.drop('price')]
y = df_ent['price']

In [54]:
X.head()

,lat,lon
id,,
3659,-34.558289,-58.443645
70676,-34.558289,-58.443645
70677,-34.558289,-58.443645
70678,-34.558289,-58.443645
70679,-34.558289,-58.443645


In [55]:
X_train, X_test, y_train, y_test = sk.model_selection.train_test_split(X, y, test_size=0.2, random_state=42)

# Definimos el valor de los hiperparámetros a usar por el modelo
n_estimators = 50
max_depth = 5

### NO CAMBIAR RandomForestRegressor por otro modelo
reg = sk.ensemble.RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, n_jobs=-1, random_state=42)

# Entrenamos el modelo
_ = reg.fit(X_train, y_train)

# Cálculo del error en entrenamiento (train)
y_pred = reg.predict(X_train)
score_train = sk.metrics.root_mean_squared_error(y_train, y_pred)

# Cálculo del error en prueba (test)
y_pred = reg.predict(X_test)
score_test  = sk.metrics.root_mean_squared_error(y_test,  y_pred)

print(f"{n_estimators=} -- {max_depth=} --> {score_train=:.2f} - {score_test=:.2f}")

n_estimators=50 -- max_depth=5 --> score_train=816017.21 - score_test=3061117.50


## 4. Solución para subir Kaggle

In [56]:
df_ap = pd.read_csv(f"{DIR}/a_predecir.csv", index_col="id")
df_ap.head(2)

,description,address,lat,lon,publication_date,publisher_id,features,location_0,location_1,location_2,location_3,location_4,operation_type,property_type,source,price,currency_type
id,,,,,,,,,,,,,,,,,
264893,casa construida en 2 plantas. capacidad para ...,"Calle Esmeralda 599, Buenos Aires, Ciudad Autó...",-34.601357,-58.378109,3 nov 2024,NaN,4 dormitorios;4 baños;308 m²;internet;balcón;c...,Argentina,Ciudad Autónoma de Buenos Aires,Buenos Aires,Ciudad Autónoma de Buenos Aires,NaN,venta,casa,properati,NaN,dolares
264899,congreso piso 4 ambientes antiguo reciclado mu...,"República Bolivariana de Venezuela 1699, Monts...",-34.614742,-58.390285,18 nov 2022,NaN,3 dormitorios;3 baños;91 m²;balcón;gas natural...,Argentina,Ciudad Autónoma de Buenos Aires,Montserrat,Ciudad de Buenos Aires,NaN,venta,casa,properati,NaN,dolares


In [57]:
df_ap.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13471 entries, 264893 to 696882
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   description       13471 non-null  object 
 1   address           13426 non-null  object 
 2   lat               3524 non-null   float64
 3   lon               3524 non-null   float64
 4   publication_date  3524 non-null   object 
 5   publisher_id      10007 non-null  float64
 6   features          13471 non-null  object 
 7   location_0        13471 non-null  object 
 8   location_1        13471 non-null  object 
 9   location_2        13471 non-null  object 
 10  location_3        13417 non-null  object 
 11  location_4        698 non-null    object 
 12  operation_type    13471 non-null  object 
 13  property_type     13471 non-null  object 
 14  source            13411 non-null  object 
 15  price             0 non-null      float64
 16  currency_type     13471 non-null  objec

In [58]:
X = df_ent[df_ent.columns.drop('price')]
y = df_ent['price']

# Entrenamos el modelo con todos los datos de entrenamiento.csv
reg.fit(X, y)

RandomForestRegressor(max_depth=5, n_estimators=50, n_jobs=-1, random_state=42)

In [59]:
# Hacemos en df_ap la misma limpieza que en df_ent
for i in ciudades_lat_lon.index:
  df_ap.loc[df_ap["location_2"] == ciudades_lat_lon.loc[i,"location_2"], "lat" ]= ciudades_lat_lon.loc[i,"lat"]
  df_ap.loc[df_ap["location_2"] == ciudades_lat_lon.loc[i,"location_2"], "lon" ]= ciudades_lat_lon.loc[i,"lon"]

In [60]:
df_ap.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13471 entries, 264893 to 696882
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   description       13471 non-null  object 
 1   address           13426 non-null  object 
 2   lat               13471 non-null  float64
 3   lon               13471 non-null  float64
 4   publication_date  3524 non-null   object 
 5   publisher_id      10007 non-null  float64
 6   features          13471 non-null  object 
 7   location_0        13471 non-null  object 
 8   location_1        13471 non-null  object 
 9   location_2        13471 non-null  object 
 10  location_3        13417 non-null  object 
 11  location_4        698 non-null    object 
 12  operation_type    13471 non-null  object 
 13  property_type     13471 non-null  object 
 14  source            13411 non-null  object 
 15  price             0 non-null      float64
 16  currency_type     13471 non-null  objec

In [61]:
df_ap = df_ap.fillna(0)

df_ap = df_ap.select_dtypes('number')

X_ap = df_ap[X.columns]

# Predecimos los precios del dataset a predecir
y_pred_ap = reg.predict(X_ap)
y_pred_ap

array([232193.94134851, 140453.9413034 , 232193.94134851, ...,
       191203.81677453, 113990.8169101 , 261548.43264834])

,lat,lon
id,,
264893,-34.601357,-58.378109
264899,-34.614742,-58.390285


In [62]:
# Lleno el precio de df_ap con las predicciones
df_ap["price"] = y_pred_ap.round(2)

# Grabo el df_ap en un archivo csv para subir a Kaggle
df_ap["price"].to_csv("/content/drive/MyDrive/UBA/Especializacion/DM/Primera-entrega/solucion-version4.csv")

In [ ]:
df_ap

,lat,lon,publisher_id,price
id,,,,
264893,-34.601357,-58.378109,0.0,138668.05
264899,-34.614742,-58.390285,0.0,138668.05
264910,-34.608849,-58.378567,0.0,138668.05
264915,-34.601772,-58.386559,0.0,138668.05
264919,-34.606712,-58.392914,0.0,138668.05
...,...,...,...,...
72571,-34.585030,-58.412045,0.0,306298.29
72713,-34.620113,-58.430485,0.0,174877.55
696861,-34.608000,-58.430000,7366.0,174046.96
